# 03 · Anomaly detection
STL residual + Isolation Forest / PCA + PELT changepoints, clustered into incidents.

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt
pd.set_option("display.max_columns", 60)
from networkanalysis.db.database import query_df, table_counts
from networkanalysis.pipeline.features import build_site_feature_table, KPI_DIRECTION, HEADLINE_KPIS

In [ ]:
from networkanalysis.analytics import anomaly, scoring
feat = build_site_feature_table()
sc, _ = scoring.compute_scorecard(feat)
uni = anomaly.univariate_anomalies(feat, scorecard=sc)
mv  = anomaly.multivariate_anomalies(feat, sc)
print(len(uni), "univariate,", len(mv), "multivariate events")

In [ ]:
# walk one anomaly: STL decomposition of a flagged site's TCP RTT
from statsmodels.tsa.seasonal import STL
site = uni.sort_values("severity", ascending=False).iloc[0].entity_id
s = feat[feat.site_id==site].set_index("ts_hour")["tcp_client_rtt_ms"]
STL(s.interpolate(), period=24, robust=True).fit().plot(); plt.suptitle(site)

In [ ]:
det = anomaly.cluster_incidents(pd.concat([uni, mv]))
det[["start_ts","end_ts","n_sites","predicted_class","match_iou","matched_incident_id"]]

In [ ]:
# recall vs ground truth
gt = query_df("SELECT incident_id, incident_class FROM dim_incident")
print(f"{(det.match_iou>0.2).sum()} / {len(gt)} ground-truth incidents matched (IoU>0.2)")